# Doc2Vec: vectorizar frases o documentos

Word2Vec aprende vectores para palabras; Doc2Vec aprende un vector para cada documento (aquí, cada frase o fila de texto). No se generan pares skip-gram manualmente: `gensim` gestiona el entrenamiento.

Cada fila/frase se tratará como un documento independiente. Si una fila contiene un documento largo, también funciona.

## 1. Instalar e importar dependencias

In [1]:
%pip install -q gensim pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 20.6 MB/s eta 0:00:00


In [13]:
import re
import io
import numpy as np
import pandas as pd
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from gensim.utils import simple_preprocess

SEED = 42
DIM_VECTORES = 128
path_to_file = ""

## 2. Cargar los datos

Ajusta `path_to_file` y `TEXT_COLUMN`. La primera celda muestra las columnas disponibles. Si el CSV tiene varias columnas, elige explícitamente la que contiene las frases; no conviene entrenar sobre el identificador u otras variables.

In [14]:
df = pd.read_csv(path_to_file)
print("Forma del dataset:", df.shape)
print("Columnas:", df.columns.tolist())
display(df.head())

Forma del dataset: (500, 3)
Columnas: ['User', 'Post', 'Label']


,User,Post,Label
0,user-0,"['Its not a viable option, and youll be leavin...",Supportive
1,user-1,['It can be hard to appreciate the notion that...,Ideation
2,user-2,"['Hi, so last night i was sitting on the ledge...",Behavior
3,user-3,['I tried to kill my self once and failed badl...,Attempt
4,user-4,['Hi NEM3030. What sorts of things do you enjo...,Ideation


In [15]:
# Cambia este valor por el nombre exacto de la columna que contiene el texto.
TEXT_COLUMN = "Post"

texts = df[TEXT_COLUMN].dropna().astype(str).tolist()
texts = [text.strip() for text in texts if text.strip()]
print(f"Textos válidos para entrenar: {len(texts)}")
print("Ejemplo:", texts[0] if texts else "No hay textos")

Textos válidos para entrenar: 500
Ejemplo: ['Its not a viable option, and youll be leaving your wife behind. Youd Pain her beyond comprehension.It sucks worrying about money, I know that first hand. It can definitely feel hopeless, as you seem to be Tired aware of. Your wife might need to chip in financially. I know time is an issue, but even 10-15 hours a Asthenia could alleviate a lot of the pressure. In the meantime, get your shit together - write that resume tomorrow. No excuses, get it done and send it out. Whether you believe in some sort of powerful being or force governing things or not, things really do work themselves out. This is a big test for you, and youll pull through. Just try to stay as positive as you can and everything will work out.']


## 3. Preprocesar y etiquetar cada frase

`simple_preprocess` pasa el texto a minúsculas y lo divide en tokens, eliminando puntuación y tokens demasiado cortos/largos.

In [16]:
tagged_documents = [
    TaggedDocument(
        words=simple_preprocess(text, deacc=True),
        tags=[f"DOC_{i}"]
    )
    for i, text in enumerate(texts)
]
tagged_documents = [doc for doc in tagged_documents if doc.words]

print("Documentos tokenizados:", len(tagged_documents))
print("Ejemplo de tokens:", tagged_documents[0].words[:30])

Documentos tokenizados: 500
Ejemplo de tokens: ['its', 'not', 'viable', 'option', 'and', 'youll', 'be', 'leaving', 'your', 'wife', 'behind', 'youd', 'pain', 'her', 'beyond', 'comprehension', 'it', 'sucks', 'worrying', 'about', 'money', 'know', 'that', 'first', 'hand', 'it', 'can', 'definitely', 'feel', 'hopeless']


## 4. Entrenar Doc2Vec

- `vector_size`: dimensión del vector de cada frase/documento.
- `epochs`: número de pasadas por el corpus.
- `window`: distancia máxima a la que 2 palabras se consideran cercanas

In [19]:
model = Doc2Vec(
    documents=tagged_documents,
    vector_size=DIM_VECTORES,
    window=5,
    min_count=2,
    workers=4,
    epochs=40,
    dm=1,
    seed=SEED
)

print("Tamaño del vocabulario:", len(model.wv))

Tamaño del vocabulario: 9470


## 5. Obtener los vectores de las frases del corpus

`model.dv` contiene los vectores aprendidos para los documentos etiquetados.

In [21]:
doc_ids = [doc.tags[0] for doc in tagged_documents]
vectors = np.vstack([model.dv[doc_id] for doc_id in doc_ids])

texts_used = [
    texts[int(doc_id.split("_")[1])]
    for doc_id in doc_ids
]

print("Matriz de vectores:", vectors.shape)  # (n_documentos, DIM_VECTORES)

Matriz de vectores: (500, 128)


## 6. Vectorizar una frase nueva

Dada una nueva frase, para vectorizarla se usa la función `infer_vector`. El modelo debe haber sido entrenado previamente.

In [27]:
new_sentence = "The wide road shimmered in the hot sun."
new_tokens = simple_preprocess(new_sentence, deacc=True)
new_vector = model.infer_vector(new_tokens, epochs=40)

print("Tamaño del vector:", new_vector.size)

Tamaño del vector: 128


## 7. Buscar documentos similares

Puedes comparar una frase nueva con los documentos etiquetados usando similitud coseno.

In [28]:
similar_docs = model.dv.most_similar([new_vector], topn=min(5, len(model.dv)))
for doc_id, score in similar_docs:
    i = int(doc_id.split("_")[1])
    print(f"{score:.3f} | {texts[i][:300]}")

0.696 | ['As someone who is feeling down in a similar situation. Killing yourself should never an option. Things can always be repaired, maybe not overnight, but nothing is worth losing your life. There are always people and organizations outside of family and friends than can give you a better perspective 
0.696 | ['Im interested, since youve pretty much described my life. Im a guy, so not so many men willing to have sex with me, but the emptiness/friends of obligation is Tired familiar. Id Tired much like to hear more of your story.', 'The apathy, too. Only reason I didnt kill myself tonight is because I pro
0.659 | ['I feel exactly the same way as youve described here. The hate and resentment I feel have become so unbearable. However, I really wish I could live... I really wanted to live Tired long... But unless I find a way to get as far away as possible from humans by the time 4 years have passed, then I wil
0.658 | ['Youre gonna have to get a new therapist. You can try talking to

## 8. Guardar el modelo, vectores y metadatos

Se crean tres archivos: el modelo Doc2Vec, una matriz TSV compatible con herramientas de visualización de embeddings y un archivo de metadatos con el texto de cada documento.

In [29]:
model.save("doc2vec.model")
np.savetxt("doc_vectors.tsv", vectors, delimiter="\t")

with open("doc_metadata.tsv", "w", encoding="utf-8") as f:
    for text in texts_used:
        # Evita saltos de línea que desalineen el archivo de metadatos
        f.write(re.sub(r"\s+", " ", text).strip() + "\n")

print("Guardados: doc2vec.model, doc_vectors.tsv, doc_metadata.tsv")

Guardados: doc2vec.model, doc_vectors.tsv, doc_metadata.tsv
